# EDA: 538 motos válidas (motos_final.json)

Dataset intermedio limpio (validado como moto). Aplica regex de marcas y filtro de precios (USD, 100 < precio < 5000). Sin gráficos.

In [1]:
import json, re, sys
import pandas as pd
from pathlib import Path

BASE = None
for nivel in ("data/processed", "../data/processed", "../../data/processed"):
    if Path(nivel).exists():
        BASE = Path(nivel).resolve()
        break
assert BASE is not None

with open(BASE / "final/motos_final.json", encoding="utf-8") as f:
    df = pd.DataFrame(json.load(f))
print("Total inicial:", len(df))

Total inicial: 538


In [2]:
MARCAS = [
    ("topmaq", "Topmaq"), ("vedca", "Vedca"), (r"\bizuki\b", "Izuki"),
    ("yoazaky|yoazaki|yoa ?zaky|yoazak", "Yoazaki"), ("mountain", "Mountain"),
    ("bucatti|bucati|buccati", "Bucatti"), ("fly racing|flywing|\bfly\b", "Fly Racing"),
    ("xmox", "Xmox"), ("mishozuki", "Mishozuki"), (r"\bava\b", "Ava"),
    ("halcon", "Halcon"), ("kvitova|kivitova|kitova", "Kvitova"),
    ("murasaki", "Murasaki"), ("panda", "Panda"), (r"\bmaf\b", "Maf"),
    (r"\bmaox\b", "Maox"), (r"\btorvan\b", "Torvan"), ("mikazuki", "Mikazuki"),
    ("dcmlz", "DCM-LZ"), ("protenzza|protenza|prontenza", "Protenzza"),
    ("voltax", "Voltax"), ("volta", "Volta"), ("meacon", "Meacon"),
    ("zuzuki|suzuki|\bzuki\b", "Zuki"), (r"\bsnopy\b", "Snopy"),
    ("yadea", "Yadea"), (r"\bxtreme\b", "Xtreme"), ("mavis", "Mavis"),
    (r"\bgt\b", "GT"),
]

def _norm(s):
    plano = re.sub(r"[^a-z0-9]+", " ", (s or "").lower())
    return re.sub(r"\s+", " ", plano).strip()

def marca_de_titulo(titulo):
    texto = _norm(titulo)
    for patron, marca in MARCAS:
        if re.search(patron, texto):
            return marca
    return None

df["marca"] = df["titulo"].map(marca_de_titulo)

In [3]:
n_init = len(df)
df_usd = df[df["moneda"] == "USD"]
print("Eliminados por no USD:", n_init - len(df_usd))
n_antes = len(df_usd)
df_pre = df_usd[df_usd["precio"] > 100]
print("Eliminados por precio <= 100:", n_antes - len(df_pre))
n_antes = len(df_pre)
df_final = df_pre[df_pre["precio"] < 5000]
print("Eliminados por precio >= 5000:", n_antes - len(df_final))
print("Total tras filtrar por USD y rango de precio:", len(df_final))
df = df_final.reset_index(drop=True)

Eliminados por no USD: 14
Eliminados por precio <= 100: 11
Eliminados por precio >= 5000: 0
Total tras filtrar por USD y rango de precio: 513


In [4]:
print("Describe completo de precios:")
print(df["precio"].describe().to_string())
print()
print("Percentiles 10, 25, 50, 75, 90:")
print(df["precio"].quantile([0.10, 0.25, 0.50, 0.75, 0.90]).to_string())

Describe completo de precios:
count     513.000000
mean     2122.643275
std       797.033861
min       111.000000
25%      1550.000000
50%      2400.000000
75%      2700.000000
max      4200.000000

Percentiles 10, 25, 50, 75, 90:
0.10     850.0
0.25    1550.0
0.50    2400.0
0.75    2700.0
0.90    3000.0


In [5]:
bins = [(0, 500), (500, 1500), (1500, 2500), (2500, 3500), (3500, float("inf"))]
etiquetas = ["<500", "500-1500", "1500-2500", "2500-3500", ">3500"]
print("Distribución por rangos de precio:")
df["rango"] = pd.cut(df["precio"], bins=[b[0] for b in bins] + [float("inf")], labels=etiquetas)
print(df["rango"].value_counts().sort_index().to_string())

Distribución por rangos de precio:
rango
<500          10
500-1500     116
1500-2500    172
2500-3500    209
>3500          6


In [6]:
top10 = df["marca"].value_counts().head(10)
print("Top 15 marcas por número de anuncios:")
print(df["marca"].value_counts().head(15).to_string())
print("\nPrecio mediano y media por marca (top 10):")
print(df[df["marca"].isin(top10.index)].groupby("marca")["precio"].agg(["median", "mean"]).sort_values("median", ascending=False).to_string())

Top 15 marcas por número de anuncios:
marca
Topmaq        58
Vedca         42
Mountain      37
Yoazaki       27
Izuki         25
Bucatti       25
Kvitova       20
Maf           13
Fly Racing    12
Ava           11
Panda         11
Mikazuki       9
Xmox           9
Mishozuki      7
DCM-LZ         6

Precio mediano y media por marca (top 10):
            median         mean
marca                          
Mountain    3000.0  2845.405405
Yoazaki     3000.0  2854.814815
Izuki       2800.0  2910.000000
Bucatti     2700.0  2680.400000
Topmaq      2650.0  2636.896552
Fly Racing  2525.0  2535.000000
Ava         2500.0  2527.272727
Vedca       1925.0  2085.476190
Kvitova     1605.0  1611.000000
Maf          780.0   992.307692


In [7]:
k = Path("src/scrapers") if Path("src/scrapers/revolico.py").exists() else Path("../src/scrapers")
sys.path.insert(0, str(k.resolve()))
import revolico as rv

listados = sorted((rv.CACHE_DIR / "motos").glob("listado_*.html"))
cat = rv.extraer_catalogos(listados[0].read_text(encoding="utf-8"))
df["provincia"] = df["provinceId"].map(cat["provincias"])
df["municipio"] = df["municipalityId"].map(cat["municipios"]).fillna(df["provincia"])

top_muni = df["municipio"].value_counts().head(10)
print("Top 15 municipios:")
print(df["municipio"].value_counts().head(15).to_string())
print("\nPrecio mediano por municipio (top 10):")
print(df[df["municipio"].isin(top_muni.index)].groupby("municipio")["precio"].median().sort_values(ascending=False).to_string())

Top 15 municipios:
municipio
Centro Habana            110
Playa                    109
Plaza                     95
Cerro                     44
Diez de Octubre           27
La Lisa                   26
Marianao                  25
Arroyo Naranjo            17
Habana Vieja              15
Boyeros                   13
Cotorro                   12
Habana del Este            8
Guanabacoa                 6
San Miguel del Padrón      4
Artemisa                   1

Precio mediano por municipio (top 10):
municipio
Habana Vieja       2600.0
Cerro              2505.0
Playa              2500.0
Diez de Octubre    2460.0
Centro Habana      2450.0
Arroyo Naranjo     2400.0
Plaza              2400.0
Marianao           2400.0
La Lisa            1750.0
Boyeros            1500.0


In [8]:
print("Anuncios con marca detectada:", int(df["marca"].notna().sum()))
print("Anuncios sin marca detectada:", int(df["marca"].isna().sum()))

Anuncios con marca detectada: 331
Anuncios sin marca detectada: 182
